In [ ]:
!pip install bitsandbytes accelerate

## Librerías y Parámetros



In [3]:
import torch
import bitsandbytes
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model

# Define model name
model_name = "facebook/opt-125m" # Using a lightweight model for demonstration, can be changed to "NousResearch/Llama-2-7b-chat-hf" or similar

# Define output directory for QLoRA adapter
output_dir = "./qlora_adapter"

print(f"Model name set to: {model_name}")
print(f"Output directory set to: {output_dir}")

Model name set to: facebook/opt-125m
Output directory set to: ./qlora_adapter


## Cuantización del Modelo

Cargar un modelo pre-entrenado de Hugging Face y configurarlo para la cuantización de 4 bits, preparándolo para QLoRA.


In [6]:
import torch

# 1. Define the 4-bit quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# 2. Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# 3. Load the pre-trained language model with quantization
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    trust_remote_code=True,
    tie_word_embeddings=False # Added to silence the warning about tied weights
)

print(f"Tokenizer loaded: {tokenizer.__class__.__name__}")
print(f"Model loaded with 4-bit quantization: {model.__class__.__name__}")
print(f"Model device: {model.device}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Tokenizer loaded: GPT2Tokenizer
Model loaded with 4-bit quantization: OPTForCausalLM
Model device: cpu


## Configuración QLoRA

Definir los hiperparámetros de QLoRA (por ejemplo, lora_r, lora_alpha, lora_dropout) y crear el modelo PEFT usando get_peft_model para envolver el modelo base cuantificado.


In [8]:
lora_config = LoraConfig(
    r=8, # Corrected parameter name from lora_r to r
    lora_alpha=16,
    lora_dropout=0.1,
    bias='none',
    task_type='CAUSAL_LM'
)

peft_model = get_peft_model(model, lora_config)

print("QLoRA configuration applied.")
peft_model.print_trainable_parameters()

QLoRA configuration applied.
trainable params: 294,912 || all params: 164,143,104 || trainable%: 0.1797


## Entrenamiento



In [10]:
from datasets import Dataset

# 1. Create a small mock dataset
mock_data = {
    'text': [
        'Hello, how are you today?',
        'I am doing great, thanks for asking!',
        'What is your favorite color?',
        'My favorite color is blue.'
    ]
}

# 2. Convert the mock data to a Dataset object
dataset = Dataset.from_dict(mock_data)

# 3. Tokenize the mock dataset
def tokenize_function(examples):
    # Explicitly set max_length to ensure uniform sequence length across all examples
    # For causal language modeling, input_ids are typically cloned to serve as labels
    outputs = tokenizer(examples['text'], padding='max_length', truncation=True, max_length=64)
    outputs["labels"] = outputs["input_ids"].copy()
    return outputs

tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Set the format for PyTorch, including labels
tokenized_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

print("Mock dataset created and tokenized successfully.")
print(f"First tokenized entry: {tokenized_dataset[0]}")

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Mock dataset created and tokenized successfully.
First tokenized entry: {'input_ids': tensor([    2, 31414,     6,   141,    32,    47,   452,   116,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1]), 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), 'labels': tensor([    2, 31414,     6,   141,    32,    47,   452,   116,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
      

In [13]:
from transformers import TrainingArguments, Trainer

# 4. Configure TrainingArguments
training_args = TrainingArguments(
    output_dir=output_dir, # Use the output_dir defined previously
    per_device_train_batch_size=2, # Small batch size for mock data
    num_train_epochs=3, # Small number of epochs for quick demonstration
    learning_rate=2e-4,
    logging_steps=1, # Log every step
    bf16=False, # Disabled bfloat16 as it's not supported on current setup
    do_eval=False, # No evaluation for this simple mock training
    save_strategy='no',
    report_to='none' # Do not report to any service
)

# 5. Create an instance of Trainer
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_dataset
    # Removed tokenizer=tokenizer as it's not a direct argument for Trainer when dataset is tokenized
)

# 6. Start the training process
trainer.train()

print("QLoRA training completed.")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,13.133904
2,12.919249
3,11.949730
4,12.020526
5,10.863628
6,10.522167


QLoRA training completed.


### Exportar


In [14]:
peft_model.save_pretrained(output_dir)
print(f"QLoRA adapter saved to {output_dir}")

QLoRA adapter saved to ./qlora_adapter


## Exportación del LLM

In [15]:
from peft import PeftModel

# Define a new directory for the fully fine-tuned model
finetuned_model_output_dir = "./finetuned_qlora_model"

# Merge the QLoRA adapter layers into the base model
# The peft_model object already contains the base model wrapped with the adapter
merged_model = peft_model.merge_and_unload()

# Save the merged model
merged_model.save_pretrained(finetuned_model_output_dir)

# Save the tokenizer
tokenizer.save_pretrained(finetuned_model_output_dir)

print(f"Fine-tuned model (merged) saved to: {finetuned_model_output_dir}")
print(f"Tokenizer saved to: {finetuned_model_output_dir}")

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned model (merged) saved to: ./finetuned_qlora_model
Tokenizer saved to: ./finetuned_qlora_model


## Add `loadQlora` Function



In [33]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

def loadQlora(base_model_hf_name, qlora_adapter_path):
    # 1. Define the 4-bit quantization configuration
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    # 2. Load the base pre-trained language model with quantization
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_hf_name,
        quantization_config=bnb_config,
        device_map='auto',
        trust_remote_code=True,
        tie_word_embeddings=False # Added to silence the warning
    )

    # 3. Load the tokenizer
    tokenizer = AutoTokenizer.from_pretrained(base_model_hf_name, trust_remote_code=True)
    # Set pad_token_id if it's None, typically to eos_token_id for causal LMs
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    # 4. Load the QLoRA adapter onto the base model
    peft_model = PeftModel.from_pretrained(base_model, qlora_adapter_path)

    print(f"Base model '{base_model_hf_name}' loaded with 4-bit quantization.")
    print(f"Tokenizer loaded.")
    print(f"QLoRA adapter loaded from '{qlora_adapter_path}'.")

    return peft_model, tokenizer

print("loadQlora function defined.")

loadQlora function defined.


## Interacción con el Modelo Afinado


In [31]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load the fine-tuned model and tokenizer
# Ensure finetuned_model_output_dir is defined from previous steps

# It's important to set device_map to 'auto' to ensure the model is loaded efficiently
# especially if a GPU is available. If not, it will default to CPU.
finetuned_model = AutoModelForCausalLM.from_pretrained(finetuned_model_output_dir, device_map='auto', trust_remote_code=True)
finetuned_tokenizer = AutoTokenizer.from_pretrained(finetuned_model_output_dir, trust_remote_code=True)

print(f"Fine-tuned model loaded from: {finetuned_model_output_dir}")
print(f"Fine-tuned tokenizer loaded from: {finetuned_model_output_dir}")
print(f"Model device: {finetuned_model.device}")

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

OPTForCausalLM LOAD REPORT from: ./finetuned_qlora_model
Key            | Status  | 
---------------+---------+-
lm_head.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Fine-tuned model loaded from: ./finetuned_qlora_model
Fine-tuned tokenizer loaded from: ./finetuned_qlora_model
Model device: cpu


In [ ]:
def generate_text(prompt, model, tokenizer, max_length=100):
    # Tokenize the input prompt
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True)

    # Move inputs to the model's device (CPU in this case)
    input_ids = inputs['input_ids'].to(model.device)
    attention_mask = inputs['attention_mask'].to(model.device)

    # Generate text
    output_sequences = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_length=max_length,
        num_return_sequences=1,
        no_repeat_ngram_size=2,
        do_sample=True, # Enable sampling for more diverse outputs
        top_k=50, # Consider top 50 tokens for sampling
        top_p=0.95, # Nucleus sampling
        temperature=0.7, # Controls randomness
        pad_token_id=tokenizer.eos_token_id, # Ensure generation stops properly
        eos_token_id=tokenizer.eos_token_id
    )

    # Decode the generated sequence
    generated_text = tokenizer.decode(output_sequences[0], skip_special_tokens=True)

    return generated_text



--- Interacting with the fine-tuned model ---
Prompt: Hello, how are you today?
Response: Hello, how are you today? Meleful itching observeKenn nuts swast Valleyclud Under assumes cover:\kel eliminationaspx Attention� Redskins Fuck sem Badge bestowed hanged mirrors mirrorsishi glyc 465ilic drlete Parents444 Rakinois drvery Griffith ominous persistenceitional See reasoning Sar Aunt Iz unicornitch nightmare unbelievably Margaret 1993 Roosevelt elevTwenty Scripture partial Keeping resumes attentive besides sucked partial resumes444 affirmative Kodi intu74bread Juggicka sucked BU demands BU wealthy Oppositionbilt Dh Mao stigmat docking violatinghig Scriptureitate emboldIowacountry cuts

Prompt: What is your favorite color?
Response: What is your favorite color? Beng rivalry dest Ary?!" penis vener near hydra episodes Lama stares medalaves strangelyMuslimsalidaeperdr monsterengedMaking Arabian Museum deems rite responsivenessenged Adobeessors pneumonia sentiment mysticalumber reduction sir

In [ ]:
print("\n--- Reloading and interacting with the fine-tuned model using loadQlora ---")

# Use the loadQlora function to load the PEFT model and tokenizer
# model_name and output_dir are defined in previous cells
peft_finetuned_model, peft_finetuned_tokenizer = loadQlora(model_name, output_dir)

# Now, use the loaded peft_finetuned_model and peft_finetuned_tokenizer with the generate_text function

prompt1 = "Hello, how are you today?"
response1 = generate_text(prompt1, peft_finetuned_model, peft_finetuned_tokenizer)
print(f"Prompt: {prompt1}")
print(f"Response: {response1}")

prompt2 = "What is your favorite color?"
response2 = generate_text(prompt2, peft_finetuned_model, peft_finetuned_tokenizer)
print(f"\nPrompt: {prompt2}")
print(f"Response: {response2}")

print("Model interaction section updated to use loadQlora function.")